In [1]:
import numpy as np 
import pandas as pd

#load data
cedar = pd.read_csv("final_predicted.csv")
cedar["epitope_id"] = cedar["epitope_id"].astype(str)
print("loaded", len(cedar), "peptides")
cedar.head()

loaded 6327 peptides


,epitope_id,best_hla,label,mt_seq,wt_seq,mt_length,mt_affinity,mt_affinity_percentile,mt_presentation_score,mt_presentation_percentile,wt_affinity,wt_affinity_percentile,wt_presentation_score,wt_presentation_percentile,mt_presented,wt_presented,delta_affinity,delta_presentation_score,delta_log_affinity,group
0,551,HLA-A*02:01,1,ACDPHSGHFV,ARDPHSGHFV,10,3125.736683,2.796875,0.053443,4.807092,13379.793366,7.050250,0.029345,8.269212,0,0,-10254.056683,0.024097,-1.454076,notpresented_immunogenic
1,785,HLA-B*44:03,0,ADVEFCLSL,ANVEFCL,9,848.828242,0.972500,0.108789,2.760054,32406.910919,40.329625,0.004118,62.744674,0,0,-31558.082677,0.104670,-3.642270,notpresented_nonimmunogenic
2,3147,HLA-A*02:01,1,AMLGTHTMEV,AMLSPHAMDV,10,37.507257,0.260250,0.768811,0.326821,104.448134,0.625750,0.483023,0.858478,1,1,-66.940877,0.285789,-1.024156,presented_immunogenic
3,14672,HLA-A*01:01,1,EVDPIGHLY,EVVPISHLY,9,33.501012,0.007125,0.983180,0.004511,164.471650,0.231000,0.941244,0.060136,1,1,-130.970638,0.041936,-1.591163,presented_immunogenic
4,23214,HLA-A*02:01,1,GVYDGEEHSV,GAYDGEEH,10,46.603713,0.330000,0.871138,0.167038,29876.517551,41.398000,0.003482,99.286603,1,0,-29829.913838,0.867656,-6.463148,presented_immunogenic


In [2]:
#make tabel to look up epitopes easily
label_lookup = {}
for row in range(len(cedar)): 
    #epitope id for row
    eid = cedar["epitope_id"].iloc[row]
    #all useful metadata
    label_lookup[eid] = {
        "label": cedar["label"].iloc[row], 
        "mt_seq": str(cedar["mt_seq"].iloc[row]), 
        "wt_seq": str(cedar["wt_seq"].iloc[row]),
        "mt_length": cedar["mt_length"].iloc[row],
        "best_hla": cedar["best_hla"].iloc[row], 
        "delta_affinity": cedar["delta_affinity"].iloc[row], 
        "delta_log_affinity": cedar["delta_log_affinity"].iloc[row], 
        "delta_presentation_score": cedar["delta_presentation_score"].iloc[row]
    }
print("unique ids", len(label_lookup))
print(label_lookup)

unique ids 6327
{'551': {'label': np.int64(1), 'mt_seq': 'ACDPHSGHFV', 'wt_seq': 'ARDPHSGHFV', 'mt_length': np.int64(10), 'best_hla': 'HLA-A*02:01', 'delta_affinity': np.float64(-10254.056682576162), 'delta_log_affinity': np.float64(-1.4540756170978677), 'delta_presentation_score': np.float64(0.024097210744201)}, '785': {'label': np.int64(0), 'mt_seq': 'ADVEFCLSL', 'wt_seq': 'ANVEFCL', 'mt_length': np.int64(9), 'best_hla': 'HLA-B*44:03', 'delta_affinity': np.float64(-31558.08267715512), 'delta_log_affinity': np.float64(-3.64227011983286), 'delta_presentation_score': np.float64(0.1046701991930356)}, '3147': {'label': np.int64(1), 'mt_seq': 'AMLGTHTMEV', 'wt_seq': 'AMLSPHAMDV', 'mt_length': np.int64(10), 'best_hla': 'HLA-A*02:01', 'delta_affinity': np.float64(-66.94087664741312), 'delta_log_affinity': np.float64(-1.0241561866805222), 'delta_presentation_score': np.float64(0.2857885057183314)}, '14672': {'label': np.int64(1), 'mt_seq': 'EVDPIGHLY', 'wt_seq': 'EVVPISHLY', 'mt_length': np.i

In [3]:
#absolute sum function
def abs_sum(x): 
    return float(np.sum(np.abs(x)))

In [5]:
#esm-2 first

all_rows = [] 

print("ESM-2 analysis")
data = np.load("esm2_final.npz", allow_pickle=True)

n = int(data["n_peptides"])
ids = [str(x) for x in data["epitope_ids"]]

#n counters
n_ok = n_no_label = n_bad_len = n_zero_mut = n_multi_mut = n_bad_embed = 0 

#read array once to speed up 
diff_eos_all = data["diff_eos"]
diff_bos_all = data["diff_bos"]

for i in range(n): 
    eid = ids[i]

    #label lookup
    if eid not in label_lookup: 
        n_no_label +=1
        continue

    info = label_lookup[eid]
    mt_seq = info["mt_seq"]
    wt_seq = info["wt_seq"]

    #find mutation
    if len(mt_seq) != len(wt_seq): 
        n_bad_len +=1
        continue 
    diffs = [j for j in range(len(mt_seq)) if mt_seq[j] != wt_seq[j]]

    if len(diffs) == 0: 
        n_zero_mut+=1
        continue 
    if len(diffs) > 1: 
        n_multi_mut += 1
        continue 
    mut_pos = diffs[0]

    #check embeds
    diff_perres = data[f"diff_perres_{i}"]
    #shape should be equal
    if diff_perres.shape[0] != len(mt_seq): 
        n_bad_embed +=1
        continue 
    
    seq_len = len(mt_seq)
    diff_eos = diff_eos_all[i]
    diff_bos = diff_bos_all[i]

    #metrics
    abs_eos = abs_sum(diff_eos)
    abs_bos = abs_sum(diff_bos)

    abs_mutpos = abs_sum(diff_perres[mut_pos])

    abs_perpos = np.array([abs_sum(diff_perres[p]) for p in range(seq_len)])
    mean_abs_all = float(abs_perpos.mean())

    top1_correct = int(np.argmax(abs_perpos) == mut_pos)
    top3_correct = int(mut_pos in np.argsort(abs_perpos)[-3:])

    #store
    all_rows.append({
        "epitope_id": eid, 
        "model": "ESM-2", 
        "label": info["label"], 
        "mt_length": info["mt_length"], 
        "best_hla": info["best_hla"], 
        "mut_pos": mut_pos + 1, #computer index starts at 0 

        "abs_sum_bos": abs_bos,
        "abs_sum_eos": abs_eos, 
        "abs_sum_mutpos": abs_mutpos, 
        "mean_abs_allpos": mean_abs_all, 

        "top1_correct": top1_correct, 
        "top3_correct": top3_correct, 

        "delta_affinity": info["delta_affinity"], 
        "delta_log_affinity": info["delta_log_affinity"], 
        "delta_presentation_score": info["delta_presentation_score"], 
    })

    n_ok +=1 
print(f"total {n}, ok={n_ok}, no_label={n_no_label}, bad_len={n_bad_len}, zero_mut={n_zero_mut}, multi_mut={n_multi_mut}, emb_bad={n_bad_embed}")

ESM-2 analysis
total 6299, ok=6201, no_label=0, bad_len=0, zero_mut=0, multi_mut=98, emb_bad=0


In [6]:
#prottrans 

print("ProtTrans analysis")
data = np.load("prottrans_final.npz", allow_pickle=True)

n = int(data["n_peptides"])
ids = [str(x) for x in data["epitope_ids"]]

#also array for speed
diff_eos_all = data["diff_eos"]

#n counters
n_ok = n_no_label = n_bad_len = n_zero_mut = n_multi_mut = n_bad_embed = 0 

for i in range(n): 
    eid = ids[i]

    #label lookup
    if eid not in label_lookup: 
        n_no_label +=1
        continue

    info = label_lookup[eid]
    mt_seq = info["mt_seq"]
    wt_seq = info["wt_seq"]

    #find mutation
    if len(mt_seq) != len(wt_seq): 
        n_bad_len +=1
        continue 
    diffs = [j for j in range(len(mt_seq)) if mt_seq[j] != wt_seq[j]]

    if len(diffs) == 0: 
        n_zero_mut+=1
        continue 
    if len(diffs) > 1: 
        n_multi_mut += 1
        continue 
    mut_pos = diffs[0]

    #check embeds
    diff_perres = data[f"diff_perres_{i}"]
    #shape should be equal
    if diff_perres.shape[0] != len(mt_seq): 
        n_bad_embed +=1
        continue 
    
    seq_len = len(mt_seq)
    diff_eos = diff_eos_all[i]

    #metrics
    abs_eos = abs_sum(diff_eos)

    abs_mutpos = abs_sum(diff_perres[mut_pos])

    abs_perpos = np.array([abs_sum(diff_perres[p]) for p in range(seq_len)])
    mean_abs_all = float(abs_perpos.mean())

    top1_correct = int(np.argmax(abs_perpos) == mut_pos)
    top3_correct = int(mut_pos in np.argsort(abs_perpos)[-3:])

    #store
    all_rows.append({
        "epitope_id": eid, 
        "model": "ProtTrans", 
        "label": info["label"], 
        "mt_length": info["mt_length"], 
        "best_hla": info["best_hla"], 
        "mut_pos": mut_pos + 1, #computer index starts at 0 

        "abs_sum_eos": abs_eos, 
        "abs_sum_mutpos": abs_mutpos, 
        "mean_abs_allpos": mean_abs_all, 

        "top1_correct": top1_correct, 
        "top3_correct": top3_correct, 

        "delta_affinity": info["delta_affinity"], 
        "delta_log_affinity": info["delta_log_affinity"], 
        "delta_presentation_score": info["delta_presentation_score"], 
    })

    n_ok +=1 
print(f"total {n}, ok={n_ok}, no_label={n_no_label}, bad_len={n_bad_len}, zero_mut={n_zero_mut}, multi_mut={n_multi_mut}, emb_bad={n_bad_embed}")

ProtTrans analysis
total 6299, ok=6201, no_label=0, bad_len=0, zero_mut=0, multi_mut=98, emb_bad=0


In [7]:
#build df to save

df = pd.DataFrame(all_rows)
out_path = "analysis_df.csv"
df.to_csv(out_path, index=False)

In [9]:
#quick summary
#is mut shift biggest residue 
print(df.groupby("model")[["top1_correct", "top3_correct"]].mean())

#do all differ by label
cols = ["abs_sum_bos", "abs_sum_eos", "abs_sum_mutpos", "mean_abs_allpos"]
print(df.groupby(["model", "label"])[cols].median())

#does mutation more when causes immunogenicity? 
print(df.groupby(["model", "label"])["top1_correct"].mean())

           top1_correct  top3_correct
model                                
ESM-2          0.999516      0.999839
ProtTrans      0.815836      0.973069
                 abs_sum_bos  abs_sum_eos  abs_sum_mutpos  mean_abs_allpos
model     label                                                           
ESM-2     0         8.941958    12.286394      106.142883        40.106445
          1         9.130219    12.469886      106.018074        40.326431
ProtTrans 0              NaN    10.290903      100.615753        53.984745
          1              NaN    10.535653      100.364655        54.408506
model      label
ESM-2      0        0.999320
           1        1.000000
ProtTrans  0        0.817873
           1        0.810826
Name: top1_correct, dtype: float64
